# Setup + Dataset

In [1]:
import torch
import numpy as np

from datasets import load_dataset,DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    DefaultDataCollator,
    TrainingArguments,
    Trainer,
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# Load SQuAD
full_dataset = load_dataset("rajpurkar/squad")

dataset = DatasetDict({
    "train": full_dataset["train"].select(range(5000)),
    "validation": full_dataset["validation"].select(range(500)),
})

print(dataset)


# Tokenizer
model_checkpoint = "distilbert/distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    model_checkpoint
)

Y:\Python\AI ML\Month 2\week 7-Deep Learning\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU: NVIDIA GeForce GTX 1650


DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 5000
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 500
    })
})


# Understand the raw QA example

In [2]:
example = dataset["train"][0]

question = example["question"]
context = example["context"]

answer_text = example["answers"]["text"][0]
answer_start = example["answers"]["answer_start"][0]
answer_end = answer_start + len(answer_text)

print("QUESTION:")
print(question)

print("\nCONTEXT:")
print(context)

print("\nANSWER:")
print(answer_text)

print("\nANSWER CHARACTER SPAN:")
print(answer_start, "→", answer_end)

print("\nVERIFY:")
print(context[answer_start:answer_end])

QUESTION:
To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?

CONTEXT:
Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.

ANSWER:
Saint Bernadette Soubirous

ANSWER CHARACTER SPAN:
515 → 541

VERIFY:
Saint Bernadette Soubirous


# SEE WHAT TOKENIZATION ACTUALLY DOES

In [3]:
inputs = tokenizer(
    question,
    context,
    max_length=384,
    truncation="only_second",
    return_offsets_mapping=True,
)

tokens = tokenizer.convert_ids_to_tokens(
    inputs["input_ids"]
)

sequence_ids = inputs.sequence_ids()
offsets = inputs["offset_mapping"]

print(
    f"{'IDX':<5}"
    f"{'TOKEN':<20}"
    f"{'SEQ_ID':<10}"
    f"{'OFFSET':<15}"
)

print("-" * 50)

for i, (token, seq_id, offset) in enumerate(
    zip(tokens, sequence_ids, offsets)
):

    print(
        f"{i:<5}"
        f"{token:<20}"
        f"{str(seq_id):<10}"
        f"{str(offset):<15}"
    )

IDX  TOKEN               SEQ_ID    OFFSET         
--------------------------------------------------
0    [CLS]               None      (0, 0)         
1    to                  0         (0, 2)         
2    whom                0         (3, 7)         
3    did                 0         (8, 11)        
4    the                 0         (12, 15)       
5    virgin              0         (16, 22)       
6    mary                0         (23, 27)       
7    allegedly           0         (28, 37)       
8    appear              0         (38, 44)       
9    in                  0         (45, 47)       
10   1858                0         (48, 52)       
11   in                  0         (53, 55)       
12   lou                 0         (56, 59)       
13   ##rdes              0         (59, 63)       
14   france              0         (64, 70)       
15   ?                   0         (70, 71)       
16   [SEP]               None      (0, 0)         
17   architectural       1     

# See the answer mapping manually

In [4]:
print("Answer text:", answer_text)
print(
    "Character span:",
    answer_start,
    "→",
    answer_end
)

print("\nContext answer:")
print(
    context[answer_start:answer_end]
)


print("\nTokens that overlap the answer:\n")

for i, (token, seq_id, offset) in enumerate(
    zip(tokens, sequence_ids, offsets)
):

    # Only context tokens
    if seq_id != 1:
        continue

    token_start, token_end = offset

    # Does this token overlap answer?
    if (
        token_start < answer_end
        and token_end > answer_start
    ):

        print(
            i,
            token,
            offset
        )

Answer text: Saint Bernadette Soubirous
Character span: 515 → 541

Context answer:
Saint Bernadette Soubirous

Tokens that overlap the answer:

130 saint (515, 520)
131 bern (521, 525)
132 ##ade (525, 528)
133 ##tte (528, 531)
134 so (532, 534)
135 ##ub (534, 536)
136 ##iro (536, 539)
137 ##us (539, 541)


# Actual preprocessing function

In [5]:
def preprocess_function(examples):

    questions = [
        q.strip()
        for q in examples["question"]
    ]

    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=384,
        truncation="only_second",
        padding="max_length",
        return_offsets_mapping=True,
    )

    offset_mapping = inputs.pop(
        "offset_mapping"
    )

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):

        answer = examples["answers"][i]

        start_char = answer["answer_start"][0]

        end_char = (
            start_char
            + len(answer["text"][0])
        )

        sequence_ids = inputs.sequence_ids(i)

        # Find beginning of context
        context_start = 0

        while sequence_ids[context_start] != 1:
            context_start += 1

        # Find end of context
        context_end = context_start

        while (
            context_end < len(sequence_ids)
            and sequence_ids[context_end] == 1
        ):
            context_end += 1

        context_end -= 1

        # Answer not present because context
        # was truncated
        if (
            offsets[context_end][1] < start_char
        ):

            start_positions.append(0)
            end_positions.append(0)

        else:

            # Find start token
            token_start = context_start

            while (
                token_start <= context_end
                and offsets[token_start][0] <= start_char
            ):
                token_start += 1

            start_positions.append(
                token_start - 1
            )

            # Find end token
            token_end = context_end

            while (
                token_end >= context_start
                and offsets[token_end][1] >= end_char
            ):
                token_end -= 1

            end_positions.append(
                token_end + 1
            )

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions

    return inputs

# Verify preprocessing before training

In [6]:
test = preprocess_function({
    "question": [question],
    "context": [context],
    "answers": [example["answers"]],
})

start_position = test["start_positions"][0]
end_position = test["end_positions"][0]

print("Start token:", start_position)
print("End token:", end_position)

print("\nStart token:")
print(tokens[start_position])

print("\nEnd token:")
print(tokens[end_position])

answer_tokens = test["input_ids"][0][
    start_position:end_position + 1
]

decoded_answer = tokenizer.decode(
    answer_tokens,
    skip_special_tokens=True
)

print("\nOriginal answer:")
print(answer_text)

print("\nDecoded answer:")
print(decoded_answer)

Start token: 130
End token: 137

Start token:
saint

End token:
##us

Original answer:
Saint Bernadette Soubirous

Decoded answer:
saint bernadette soubirous


# Process dataset + model

In [7]:
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

print(tokenized_dataset)

data_collator = DefaultDataCollator()


model = AutoModelForQuestionAnswering.from_pretrained(
    model_checkpoint
)

model.to(device)

print(
    "Model device:",
    next(model.parameters()).device
)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
        num_rows: 5000
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
        num_rows: 500
    })
})


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 14592.44it/s]
[transformers] DistilBertForQuestionAnswering LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
qa_outputs.weight       | MISSING    | 
qa_outputs.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model device: cuda:0


# Train + evaluate

In [8]:
training_args = TrainingArguments(
    output_dir="./qa-distilbert",

    learning_rate=2e-5,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    num_train_epochs=2,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    report_to="none",

    seed=42,
)


trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],

    processing_class=tokenizer,

    data_collator=data_collator,
)


trainer.train()

results = trainer.evaluate()

print(results)

Epoch,Training Loss,Validation Loss
1,3.227342,1.753178
2,1.626782,1.670119


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.92it/s]


Training Loss,Validation Loss,Epoch
1.626782,1.670119,2


{'eval_loss': 1.6701186895370483}


In [9]:
question = "What does a question answering model predict?"

context = """
A question answering model predicts the
start and end positions of an answer span
inside a given context.
"""

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
model.eval()

inputs = tokenizer(
    question,
    context,
    return_tensors="pt"
)

inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

with torch.no_grad():
    outputs = model(**inputs)

start = outputs.start_logits.argmax(dim=-1).item()
end = outputs.end_logits.argmax(dim=-1).item()

answer = tokenizer.decode(
    inputs["input_ids"][0][start:end + 1],
    skip_special_tokens=True
)

print("Answer:", answer)

Answer: the start and end positions
